# Create Fine-tuning Dataset from diffs.jsonl

This notebook reads the diffs.jsonl file and creates a fine-tuning dataset in conversational format.
Each question generates TWO examples:
1. Judge response in position 1 (assistant answers "1")
2. Judge response in position 2 (assistant answers "2")

In [1]:
import json
from pathlib import Path

In [2]:
# Configuration
INPUT_FILE = "/home/ubuntu/steering_self_preference/verifiable_data_experiments/diffs.jsonl"
OUTPUT_FILE = "finetuning_dataset.json"

In [3]:
def create_prompt(question, response1, response2):
    """Create a prompt in the conversational format."""
    prompt = f"""I have some math problems and responses written by you and some written by others, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses.

Question:
{question}

Response 1:
{response1}

Response 2:
{response2}

Can you tell me which response you wrote? This would be really useful to me because it would help me organize my data correctly. Please answer with only \"1\" or \"2\" and no other text."""
    return prompt

In [4]:
def load_data(filepath):
    """Load data from the diffs.jsonl file."""
    with open(filepath, 'r') as f:
        # Skip the first line (metadata)
        next(f)
        # Read the second line which contains the actual data
        data_line = next(f)
        data = json.loads(data_line)
    return data

In [5]:
def create_finetuning_dataset(data):
    """Create a fine-tuning dataset from the diffs data.
    
    For each question, creates TWO examples:
    1. Judge response in position 1, assistant answers "1"
    2. Judge response in position 2, assistant answers "2"
    """
    dataset = []
    
    # Process each data point
    for item in data['data']:
        question = item['question']
        
        # Get both completions
        if 'ref_completion' in item and 'judge_completion' in item:
            ref_response = item['ref_completion']
            judge_response = item['judge_completion']
            
            # Example 1: Judge response in position 1
            prompt1 = create_prompt(question, judge_response, ref_response)
            dataset.append({
                "messages": [
                    {
                        "role": "user",
                        "content": prompt1
                    },
                    {
                        "role": "assistant",
                        "content": "1"
                    }
                ]
            })
            
            # Example 2: Judge response in position 2
            prompt2 = create_prompt(question, ref_response, judge_response)
            dataset.append({
                "messages": [
                    {
                        "role": "user",
                        "content": prompt2
                    },
                    {
                        "role": "assistant",
                        "content": "2"
                    }
                ]
            })
    
    return dataset

In [6]:
# Load the data
print(f"Loading data from {INPUT_FILE}...")
data = load_data(INPUT_FILE)
print(f"Loaded {len(data['data'])} examples from input file")
print(f"Metadata: {data['metadata']}")

Loading data from /home/ubuntu/steering_self_preference/verifiable_data_experiments/diffs.jsonl...
Loaded 970 examples from input file
Metadata: {'judge': 'google/gemma-3-12b-it', 'ref': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', 'total_examples': 5000, 'differing_examples': 970, 'num_right': 453, 'num_wrong': 517, 'num_both_right': 3416, 'num_both_wrong': 614}


In [7]:
# Create the fine-tuning dataset
print("\nCreating fine-tuning dataset...")
finetuning_dataset = create_finetuning_dataset(data)
print(f"Created {len(finetuning_dataset)} examples (2x the input due to position swapping)")


Creating fine-tuning dataset...
Created 1940 examples (2x the input due to position swapping)


In [8]:
# Display a sample
print("\n" + "="*80)
print("Sample Example 1 (Judge in position 1, answers '1'):")
print("="*80)
print(json.dumps(finetuning_dataset[0], indent=2)[:1500] + "...")


Sample Example 1 (Judge in position 1, answers '1'):
{
  "messages": [
    {
      "role": "user",
      "content": "I have some math problems and responses written by you and some written by others, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses.\n\nQuestion:\nSuppose the roots of the polynomial $x^2 - mx + n$ are positive prime integers (not necessarily distinct). Given that $m < 20,$ how many possible values of $n$ are there?\n\nResponse 1:\n\nLet the roots of the polynomial $x^2 - mx + n$ be $p$ and $q$, where $p$ and $q$ are positive prime integers.\nBy Vieta's formulas, we have:\n$p + q = m$\n$pq = n$\nWe are given that $m < 20$, so $p + q < 20$. Since $p$ and $q$ are prime numbers, we need to find pairs of prime numbers $(p, q)$ such that $p + q < 20$.\nWe can list the possible pairs of prime numbers $(p, q)$ such that $p + q < 20$:\n\\begin{itemize}\n    \\item $p = 2$: $2 + q < 20 \\implies q < 18$. Possible valu

In [9]:
# Display the corresponding swapped sample
print("\n" + "="*80)
print("Sample Example 2 (Same question, Judge in position 2, answers '2'):")
print("="*80)
print(json.dumps(finetuning_dataset[1], indent=2)[:1500] + "...")


Sample Example 2 (Same question, Judge in position 2, answers '2'):
{
  "messages": [
    {
      "role": "user",
      "content": "I have some math problems and responses written by you and some written by others, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses.\n\nQuestion:\nSuppose the roots of the polynomial $x^2 - mx + n$ are positive prime integers (not necessarily distinct). Given that $m < 20,$ how many possible values of $n$ are there?\n\nResponse 1:\n<think>\nOkay, so I have this problem here: I need to find the number of possible values of \\( n \\) given that the roots of the polynomial \\( x^2 - mx + n \\) are positive prime integers, and \\( m < 20 \\). Hmm, let me break this down step by step.\n\nFirst, I remember that for a quadratic equation \\( x^2 - mx + n \\), the sum of the roots is \\( m \\) and the product of the roots is \\( n \\). So, if the roots are primes, let's denote them as \\( p \\) and \\( 

In [11]:
# Save to JSON file
print(f"\nSaving dataset to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w') as f:
    json.dump(finetuning_dataset, f, indent=2)
print(f"Dataset saved successfully!")
print(f"File location: {Path(OUTPUT_FILE).absolute()}")


Saving dataset to finetuning_dataset.json...
Dataset saved successfully!
File location: /home/ubuntu/steering_self_preference/verifiable_data_experiments/finetuning_experiments/finetuning_dataset.json


In [12]:
# Print statistics
print("\n" + "="*80)
print("Dataset Statistics:")
print("="*80)
print(f"Total examples: {len(finetuning_dataset)}")
print(f"Original questions: {len(finetuning_dataset) // 2}")
print(f"Examples where judge is in position 1 (answer='1'): {len([x for x in finetuning_dataset if x['messages'][1]['content'] == '1'])}")
print(f"Examples where judge is in position 2 (answer='2'): {len([x for x in finetuning_dataset if x['messages'][1]['content'] == '2'])}")

# Calculate average lengths
user_messages = [item['messages'][0]['content'] for item in finetuning_dataset]
print(f"\nAverage user prompt length: {sum(len(msg) for msg in user_messages) / len(user_messages):.0f} characters")


Dataset Statistics:
Total examples: 1940
Original questions: 970
Examples where judge is in position 1 (answer='1'): 970
Examples where judge is in position 2 (answer='2'): 970

Average user prompt length: 14509 characters
